## 1. Schema


In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

project_root_text = str(PROJECT_ROOT)
if project_root_text in sys.path:
    sys.path.remove(project_root_text)
sys.path.insert(0, project_root_text)
importlib.invalidate_caches()

import safe_speed_schema
import safe_speed_data
import safe_speed_eda
import safe_speed_risk
import vru_evidence

safe_speed_schema = importlib.reload(safe_speed_schema)
safe_speed_data = importlib.reload(safe_speed_data)
safe_speed_eda = importlib.reload(safe_speed_eda)
safe_speed_risk = importlib.reload(safe_speed_risk)
vru_evidence = importlib.reload(vru_evidence)

schema_rows = safe_speed_schema.schema_rows
filter_valid_analysis_rows = safe_speed_data.filter_valid_analysis_rows
load_roads = safe_speed_data.load_roads
validate_schema = safe_speed_data.validate_schema
column_overview = safe_speed_eda.column_overview
duplicate_id_summary = safe_speed_eda.duplicate_id_summary
numeric_summary = safe_speed_eda.numeric_summary
speed_field_preview = safe_speed_eda.speed_field_preview
value_counts_table = safe_speed_eda.value_counts_table
DEFAULT_PRIORITY_WEIGHTS = safe_speed_risk.DEFAULT_PRIORITY_WEIGHTS
add_priority_score = safe_speed_risk.add_priority_score
add_speed_risk_metrics = safe_speed_risk.add_speed_risk_metrics
export_priority_segments = safe_speed_risk.export_priority_segments
summarize_speed_risk = safe_speed_risk.summarize_speed_risk
add_street_image_query_points = vru_evidence.add_street_image_query_points
add_vru_exposure_features = vru_evidence.add_vru_exposure_features
build_mapillary_vru_evidence_cache = vru_evidence.build_mapillary_vru_evidence_cache
build_segment_vru_check = vru_evidence.build_segment_vru_check
DEFAULT_VRU_EXPOSURE_WEIGHTS = vru_evidence.DEFAULT_VRU_EXPOSURE_WEIGHTS
setup_mapillary_client = vru_evidence.setup_mapillary_client
vru_status_summary = vru_evidence.vru_status_summary

display(pd.DataFrame(schema_rows()))


,field,description
0,OBJECTID,Unique ID created by GIS software
1,english_ro,English street name
2,OvertureID,Unique ID created by GIS software
3,SampleSize_avg,Average sample size if multiple TomTom dataset...
4,RoadLength,Ignore; use Shape_Length
5,WeightedSample,Average annual traffic weighted value by road ...
6,SampleSize Percent_,Ignore
7,Percentile,Travel percentile for this road
8,SpeedLimit,"Speed limit obtained by TomTom, not validated"
9,RoadClass,Overture road class


## 2. Load The Data


In [2]:
GEOJSON_PATH = DATA_DIR / 'ADB_Innovation_Thailand.geojson'

features, roads_raw = load_roads(GEOJSON_PATH)
if roads_raw.empty:
    roads = pd.DataFrame()
    print(f'Place the challenge GeoJSON at: {GEOJSON_PATH}')
else:
    roads = roads_raw.copy()
    print(f'Loaded {len(roads_raw):,} road segments from {GEOJSON_PATH}')
    display(roads_raw.head())


Loaded 55,884 road segments from /Users/markjaysondoma/Documents/safe-speed-adb-challenge/data/ADB_Innovation_Thailand.geojson


,OBJECTID,english_ro,OvertureID,SampleSize_avg,RoadLength,WeightedSample,Percent_,Percentile,SpeedLimit,RoadClass,NumberOverLimit,MedianSpeed,F85thPercentileSpeed,ForAnalysis,ProvinceID,SpeedLimitFloor,PercentOverLimit,InvPercentile,AnalysisStatus,RankedPercentile,StreetImageLink,LandUse,NO_OF_Result_Segments,PercentileBand,SampleSizeTotal,Shape_Length,_geometry_type,_coordinate_count
0,1,Surin Ring Road,1,NaN,4.632086,NaN,NaN,NaN,NaN,primary,NaN,NaN,NaN,NaN,32,NaN,NaN,NaN,Not Included,NaN,"103.4699393,14.839138,103.43891053,14.86699584",NaN,NaN,NaN,NaN,4632.086492,LineString,52
1,2,Surin Ring Road,2,44479.6,2.300000,102303.08,0.000002,0.002591,66.0,primary,128017.0,78.38,115.2,66.0,32,60.0,0.575621,0.997409,Valid,26.253458,"103.43891053,14.86699584,103.48332214,14.94166589",RURAL,1.0,0-5%,222398.0,11672.918346,LineString,74
2,3,Surin Ring Road,3,NaN,0.122010,NaN,NaN,NaN,NaN,primary,NaN,NaN,NaN,NaN,32,NaN,NaN,NaN,Not Included,NaN,"103.48332214,14.94166589,103.48440114,14.94199564",NaN,NaN,NaN,NaN,122.010111,LineString,3
3,4,Surin Ring Road,4,NaN,2.045467,NaN,NaN,NaN,NaN,primary,NaN,NaN,NaN,NaN,32,NaN,NaN,NaN,Not Included,NaN,"103.48440114,14.94199564,103.50325276,14.94199566",NaN,NaN,NaN,NaN,2045.467119,LineString,17
4,5,Surin Ring Road,5,NaN,2.028143,NaN,NaN,NaN,NaN,primary,NaN,NaN,NaN,NaN,32,NaN,NaN,NaN,Not Included,NaN,"103.50325276,14.94199566,103.5216215,14.9380875",NaN,NaN,NaN,NaN,2028.142985,LineString,8


## 3. EDA


In [3]:
if roads_raw.empty:
    print('EDA will run after GEOJSON_PATH points to a real file.')
else:
    print(f'Rows / road segments: {len(roads_raw):,}')
    print(f'Columns: {len(roads_raw.columns):,}')
    print(f'Raw GeoJSON features: {len(features):,}')

    display(column_overview(roads_raw))

    for col in ['_geometry_type', 'RoadClass', 'LandUse', 'PercentileBand', 'AnalysisStatus']:
        table = value_counts_table(roads_raw, col)
        if not table.empty:
            print(f'Top values: {col}')
            display(table)

    display(numeric_summary(roads_raw))
    display(duplicate_id_summary(roads_raw))
    display(speed_field_preview(roads_raw))


Rows / road segments: 55,884
Columns: 28
Raw GeoJSON features: 55,884


,dtype,non_null,missing,missing_pct,unique_values
RankedPercentile,float64,11544,44340,79.34,11544
Percentile,float64,11544,44340,79.34,11531
InvPercentile,float64,11544,44340,79.34,11531
WeightedSample,float64,11544,44340,79.34,11519
Percent_,float64,11544,44340,79.34,11519
SampleSize_avg,float64,11544,44340,79.34,11475
SampleSizeTotal,float64,11544,44340,79.34,11444
NumberOverLimit,float64,11544,44340,79.34,7613
PercentOverLimit,float64,11544,44340,79.34,7227
MedianSpeed,float64,11544,44340,79.34,2349


Top values: _geometry_type


,_geometry_type,segments
0,LineString,55884


Top values: RoadClass


...

Top values: AnalysisStatus


,AnalysisStatus,segments
0,Not Included,44340
1,Valid,11544


,count,mean,std,min,5%,25%,50%,75%,95%,max
SampleSize_avg,11544.0,6.697560e+05,2.976403e+06,3.333333,13679.275000,68135.287500,177238.000000,4.455252e+05,2.207437e+06,9.031018e+07
RoadLength,55884.0,1.374297e+00,4.605329e+00,0.000000,0.011500,0.029177,0.159897,7.866035e-01,6.204380e+00,1.466000e+02
WeightedSample,11544.0,4.538416e+06,3.892524e+07,0.000000,15859.620000,95435.237500,310401.700000,1.079965e+06,1.022300e+07,1.551703e+09
Percentile,11544.0,4.160893e-02,7.969394e-02,0.000000,0.000092,0.002308,0.012428,4.482255e-02,1.794565e-01,1.000000e+00
SpeedLimit,11596.0,8.053337e+01,1.843625e+01,0.000000,40.000000,80.000000,85.000000,9.000000e+01,9.000000e+01,1.200000e+02
NumberOverLimit,11544.0,2.114311e+05,1.271425e+06,0.000000,0.000000,0.000000,20466.500000,1.180442e+05,7.007855e+05,5.893935e+07
MedianSpeed,11544.0,6.454128e+01,2.123504e+01,0.000000,24.400000,53.718750,67.900000,7.940000e+01,9.250000e+01,1.300000e+02
F85thPercentileSpeed,11544.0,7.828160e+01,2.380317e+01,0.000000,37.000000,67.000000,82.000000,9.450000e+01,1.085000e+02,1.500000e+02
SpeedLimitFloor,11596.0,8.027596e+01,1.844006e+01,0.000000,40.000000,80.000000,80.000000,9.000000e+01,9.000000e+01,1.200000e+02
PercentOverLimit,11544.0,2.181039e-01,2.469572e-01,0.000000,0.000000,0.000000,0.125001,3.749988e-01,7.000002e-01,1.000000e+00


,field,duplicate_rows
0,OBJECTID,0
1,OvertureID,0


,SpeedLimit,MedianSpeed,F85thPercentileSpeed,PercentOverLimit
0,NaN,NaN,NaN,NaN
1,66.0,78.380000,115.200000,0.575621
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
5,70.0,56.766667,84.666667,0.529277
6,30.0,15.450000,21.000000,0.300000
7,NaN,NaN,NaN,NaN
8,90.0,80.925000,97.250000,0.351819
9,NaN,NaN,NaN,NaN


## 4. Filter


In [4]:
roads = filter_valid_analysis_rows(roads_raw)

if roads_raw.empty:
    print('Valid-row filter will run after the GeoJSON is loaded.')
elif 'AnalysisStatus' in roads_raw.columns:
    print(f'Filtered working dataset to AnalysisStatus == Valid: {len(roads):,} of {len(roads_raw):,} road segments')
else:
    print('AnalysisStatus column not found; continuing with all loaded road segments')

display(validate_schema(roads))
display(roads.head())


Filtered working dataset to AnalysisStatus == Valid: 11,544 of 55,884 road segments


,field,present,missing_values,non_null_values,description
0,OBJECTID,True,0,11544,Unique ID created by GIS software
1,english_ro,True,0,11544,English street name
2,OvertureID,True,0,11544,Unique ID created by GIS software
3,WeightedSample,True,0,11544,Average annual traffic weighted value by road ...
4,Percentile,True,0,11544,Travel percentile for this road
5,PercentileBand,True,0,11544,Percentile band for dashboard
6,SpeedLimit,True,0,11544,"Speed limit obtained by TomTom, not validated"
7,RoadClass,True,0,11544,Overture road class
8,NumberOverLimit,True,0,11544,Estimated annual vehicles over the speed limit...
9,MedianSpeed,True,0,11544,Median speed


,OBJECTID,english_ro,OvertureID,SampleSize_avg,RoadLength,WeightedSample,Percent_,Percentile,SpeedLimit,RoadClass,NumberOverLimit,MedianSpeed,F85thPercentileSpeed,ForAnalysis,ProvinceID,SpeedLimitFloor,PercentOverLimit,InvPercentile,AnalysisStatus,RankedPercentile,StreetImageLink,LandUse,NO_OF_Result_Segments,PercentileBand,SampleSizeTotal,Shape_Length,_geometry_type,_coordinate_count
1,2,Surin Ring Road,2,44479.600000,2.3,1.023031e+05,1.952667e-06,2.591373e-03,66.0,primary,128017.0,78.380000,115.200000,66.0,32,60.0,0.575621,0.997409,Valid,26.253458,"103.43891053,14.86699584,103.48332214,14.94166589",RURAL,1.0,0-5%,222398.0,11672.918346,LineString,74
5,6,Surin Ring Road,6,27729.666667,1.9,5.268637e+04,1.005629e-06,9.392420e-04,70.0,primary,44030.0,56.766667,84.666667,70.0,32,70.0,0.529277,0.999061,Valid,16.459198,"103.5215639,14.9379571,103.50238738,14.94199565",URBAN,1.0,0-5%,83189.0,2116.242153,LineString,12
6,7,Surin Ring Road,7,3.333333,1.9,6.333333e+00,1.208848e-10,1.208848e-10,30.0,primary,3.0,15.450000,21.000000,30.0,32,30.0,0.300000,1.000000,Valid,0.129668,"103.50238738,14.94199565,103.48521899,14.94199565",RURAL,1.0,0-5%,10.0,1859.429082,LineString,16
8,9,Surin Ring Road,9,275650.000000,3.7,1.019905e+06,1.946700e-05,4.299693e-02,90.0,primary,96979.0,80.925000,97.250000,90.0,32,90.0,0.351819,0.957003,Valid,74.057746,"103.48332214,14.94148244,103.43913438,14.86699584",RURAL,4.0,0-5%,275650.0,11643.835851,LineString,82
17,18,Sukhaphiban 5 Road,18,744341.000000,0.2,1.488682e+05,2.841458e-06,4.771872e-03,80.0,secondary,66991.0,63.600000,76.000000,80.0,10,80.0,0.090000,0.995228,Valid,34.180498,"100.6520076,13.8817104,100.6554489,13.887161",URBAN,1.0,0-5%,744341.0,708.531851,LineString,6


## 5. Compute Speed Gap


In [5]:
roads_scored = add_speed_risk_metrics(roads) if not roads.empty else roads.copy()

risk_columns = [
    'OBJECTID', 'english_ro', 'RoadClass', 'LandUse', 'SpeedLimit', 'MedianSpeed',
    'F85thPercentileSpeed', 'speed_gap_85th', 'PercentOverLimit', 'WeightedSample',
    'weighted_over_limit_pressure', 'over_limit_per_km', 'PercentileBand'
]

if roads_scored.empty:
    print('Risk metrics will populate after loading the GeoJSON.')
else:
    display(roads_scored[[c for c in risk_columns if c in roads_scored.columns]].head())
    display(summarize_speed_risk(roads_scored))


,OBJECTID,english_ro,RoadClass,LandUse,SpeedLimit,MedianSpeed,F85thPercentileSpeed,speed_gap_85th,PercentOverLimit,WeightedSample,weighted_over_limit_pressure,over_limit_per_km,PercentileBand
1,2,Surin Ring Road,primary,RURAL,66.0,78.380000,115.200000,49.200000,0.575621,1.023031e+05,58887.820000,10967.008952,0-5%
5,6,Surin Ring Road,primary,URBAN,70.0,56.766667,84.666667,14.666667,0.529277,5.268637e+04,27885.666667,20805.747556,0-5%
6,7,Surin Ring Road,primary,RURAL,30.0,15.450000,21.000000,-9.000000,0.300000,6.333333e+00,1.900000,1.613398,0-5%
8,9,Surin Ring Road,primary,RURAL,90.0,80.925000,97.250000,7.250000,0.351819,1.019905e+06,358822.300000,8328.784538,0-5%
17,18,Sukhaphiban 5 Road,secondary,URBAN,80.0,63.600000,76.000000,-4.000000,0.090000,1.488682e+05,13398.200000,94549.031076,0-5%


,RoadClass,LandUse,segment_count,total_shape_length_m,total_weighted_sample,avg_percent_over_limit,median_85th_speed_gap,total_weighted_over_limit_pressure,median_over_limit_per_km
7,trunk,URBAN,962,5.359203e+06,1.774063e+10,0.344631,8.0,6.049338e+09,65532.813474
6,trunk,RURAL,1123,9.230296e+06,1.079335e+10,0.454561,15.0,5.337844e+09,58053.638411
1,motorway,URBAN,115,8.756161e+05,9.569434e+09,0.370483,6.5,3.266873e+09,289320.958728
3,primary,URBAN,1823,4.439156e+06,5.046106e+09,0.197074,-2.0,1.021237e+09,20879.808351
2,primary,RURAL,1489,7.894117e+06,2.888823e+09,0.354602,9.0,1.010103e+09,35575.091385
5,secondary,URBAN,2861,6.275834e+06,4.197751e+09,0.103917,-11.5,3.114265e+08,0.000000
4,secondary,RURAL,3144,2.639130e+07,1.387212e+09,0.141372,-5.0,2.649008e+08,810.155090
0,motorway,RURAL,27,1.515442e+05,7.681705e+08,0.152933,0.0,1.960397e+08,28193.468671


## 6. Prioritised Segment


In [6]:
PRIORITY_WEIGHTS = DEFAULT_PRIORITY_WEIGHTS.copy()
# Edit these values to tune the prioritization model. The score is normalized by the active weight total.
PRIORITY_WEIGHTS.update({
    'weighted_over_limit_pressure': 0.45,
    'speed_gap_85th': 0.30,
    'PercentOverLimit': 0.15,
    'WeightedSample': 0.10,
})

prioritized = add_priority_score(roads_scored, weights=PRIORITY_WEIGHTS) if not roads_scored.empty else roads_scored.copy()

if prioritized.empty:
    print('Priority ranking will run after data is loaded.')
else:
    display(pd.DataFrame(PRIORITY_WEIGHTS.items(), columns=['factor', 'weight']))
    top_cols = [c for c in [
        'priority_score', 'OBJECTID', 'english_ro', 'RoadClass', 'LandUse', 'SpeedLimit',
        'F85thPercentileSpeed', 'speed_gap_85th', 'PercentOverLimit', 'WeightedSample',
        'weighted_over_limit_pressure', 'PercentileBand', 'StreetImageLink'
    ] if c in prioritized.columns]
    display(prioritized.sort_values('priority_score', ascending=False)[top_cols].head(25))


,factor,weight
0,weighted_over_limit_pressure,0.45
1,speed_gap_85th,0.30
2,PercentOverLimit,0.15
3,WeightedSample,0.10


,priority_score,OBJECTID,english_ro,RoadClass,LandUse,SpeedLimit,F85thPercentileSpeed,speed_gap_85th,PercentOverLimit,WeightedSample,weighted_over_limit_pressure,PercentileBand,StreetImageLink
15806,0.990120,15807,Udon Ratthaya Expressway,motorway,URBAN,80.0,115.000000,35.000000,0.854300,2.107923e+08,1.800800e+08,50-55%,"100.5432377,13.9018303,100.5565652,14.1450103"
16366,0.987110,16367,Prachim Ratthaya Expressway,motorway,URBAN,80.0,114.500000,34.500000,0.858595,5.437854e+07,4.668911e+07,35-40%,"100.4258304,13.799194,100.5495037,13.8097026"
16368,0.984893,16369,Prachim Ratthaya Expressway,motorway,URBAN,80.0,114.833333,34.833333,0.854856,3.158095e+07,2.699715e+07,25-30%,"100.5499176,13.8112534,100.427932,13.7984935"
177,0.984611,178,Buraphawithi Expressway,motorway,URBAN,80.0,109.947368,29.947368,0.727353,7.998745e+08,5.817913e+08,80-85%,"101.00488587,13.48893451,100.6639827,13.6596045"
36077,0.984031,36078,Phetkasem Road,trunk,RURAL,50.0,97.333333,47.333333,0.764385,2.544013e+07,1.944604e+07,25-30%,"98.4261369,8.3338412,98.50832367,8.42969965"
17518,0.983472,17519,Asian Highway,trunk,URBAN,80.0,112.333333,32.333333,0.761083,4.778082e+07,3.636516e+07,30-35%,"100.35200968,14.99199584,100.31325944,15.07532886"
35514,0.982662,35515,Chalong Rat Expressway,motorway,URBAN,80.0,108.666667,28.666667,0.714541,4.014564e+08,2.868570e+08,70-75%,"100.6890981,13.8974356,100.5940725,13.7050315"
15807,0.982298,15808,Udon Ratthaya Expressway,motorway,URBAN,80.0,106.800000,26.800000,0.822044,1.546902e+08,1.271621e+08,45-50%,"100.5569967,14.1452312,100.5439244,13.9048539"
32458,0.982136,32459,,motorway,URBAN,60.0,110.000000,50.000000,1.000000,1.066712e+07,1.066712e+07,15-20%,"100.6714758,13.655475,100.60495207,13.6749074"
17497,0.980219,17498,Asian Highway,trunk,URBAN,80.0,112.500000,32.500000,0.814579,2.082248e+07,1.696155e+07,20-25%,"100.31363238,15.07532886,100.35233777,14.99199584"


## 7. Get Coordinates


In [7]:
prioritized = add_street_image_query_points(prioritized) if not prioritized.empty else prioritized

if prioritized.empty:
    print('Street-image query points will be parsed after priority scoring runs.')
else:
    display(prioritized[['OBJECTID', 'StreetImageLink', 'query_lon', 'query_lat', 'street_image_point_count']].head())


,OBJECTID,StreetImageLink,query_lon,query_lat,street_image_point_count
1,2,"103.43891053,14.86699584,103.48332214,14.94166589",103.438911,14.866996,2
5,6,"103.5215639,14.9379571,103.50238738,14.94199565",103.521564,14.937957,2
6,7,"103.50238738,14.94199565,103.48521899,14.94199565",103.502387,14.941996,2
8,9,"103.48332214,14.94148244,103.43913438,14.86699584",103.483322,14.941482,2
17,18,"100.6520076,13.8817104,100.6554489,13.887161",100.652008,13.881710,2


## 8. VRU Evidencing Using Mapillary


In [8]:
VRU_EVIDENCE_PATH = DATA_DIR / 'mapillary_vru_evidence.csv'
# Build or resume the Mapillary evidence cache. Existing OBJECTIDs are skipped when overwrite is False.
BUILD_MAPILLARY_CACHE = True
OVERWRITE_MAPILLARY_CACHE = False
# None targets all prioritized segments. Use an integer like 5 for a quick test.
MAPILLARY_SEGMENT_LIMIT = None
# None processes all remaining coordinate-eligible segments. Use an integer like 250 for smaller batches.
MAPILLARY_BATCH_SIZE = None
MAPILLARY_CACHE_TOP_N = len(prioritized) if MAPILLARY_SEGMENT_LIMIT is None else MAPILLARY_SEGMENT_LIMIT
MAPILLARY_SEARCH_RADIUS_M = 100
# Only segments with query_lon/query_lat can be checked through Mapillary.
MAPILLARY_SKIP_MISSING_COORDS = True
# Keep this low for full-dataset runs. Set to None only when you want every Mapillary image.
MAPILLARY_IMAGES_PER_SEGMENT = 1
# Thumbnail URLs add per-image calls; keep off for quick evidence runs.
MAPILLARY_INCLUDE_THUMBNAILS = False
# Segment-level Mapillary features/traffic signs add API calls; keep True for VRU/school/crossing evidence.
USE_MAPILLARY_API_FILTERS = True
# Per-image detections can be slow. Turn this on only for deeper evidence review.
USE_MAPILLARY_IMAGE_DETECTIONS = False
MAPILLARY_DETECTION_IMAGES_PER_SEGMENT = 1
# Parallel API calls. Keep this modest to avoid rate limits; try 4, 6, or 8.
MAPILLARY_MAX_WORKERS = 6
MAPILLARY_FLUSH_EVERY = 25
MAPILLARY_PROGRESS_EVERY = 25

mapillary_target = prioritized.sort_values('priority_score', ascending=False).head(MAPILLARY_CACHE_TOP_N) if 'priority_score' in prioritized.columns else prioritized.head(MAPILLARY_CACHE_TOP_N)
mapillary_coord_ready = mapillary_target['query_lon'].notna() & mapillary_target['query_lat'].notna()
print(f'Mapillary coordinate-eligible segments: {int(mapillary_coord_ready.sum())} of {len(mapillary_target)} target segment(s)')

mly = setup_mapillary_client(PROJECT_ROOT)
print(
    f'Mapillary run config: target_segments={MAPILLARY_CACHE_TOP_N}, '
    f'batch_size={MAPILLARY_BATCH_SIZE}, '
    f'images_per_segment={MAPILLARY_IMAGES_PER_SEGMENT}, '
    f'thumbnails={MAPILLARY_INCLUDE_THUMBNAILS}, '
    f'image_detections={USE_MAPILLARY_IMAGE_DETECTIONS}, '
    f'workers={MAPILLARY_MAX_WORKERS}'
)

if BUILD_MAPILLARY_CACHE:
    cache = build_mapillary_vru_evidence_cache(
        prioritized,
        evidence_path=VRU_EVIDENCE_PATH,
        mly=mly,
        top_n=MAPILLARY_CACHE_TOP_N,
        radius_m=MAPILLARY_SEARCH_RADIUS_M,
        max_images_per_segment=MAPILLARY_IMAGES_PER_SEGMENT,
        include_thumbnails=MAPILLARY_INCLUDE_THUMBNAILS,
        use_mapillary_api_filters=USE_MAPILLARY_API_FILTERS,
        use_image_detections=USE_MAPILLARY_IMAGE_DETECTIONS,
        max_detection_images_per_segment=MAPILLARY_DETECTION_IMAGES_PER_SEGMENT,
        overwrite=OVERWRITE_MAPILLARY_CACHE,
        batch_size=MAPILLARY_BATCH_SIZE,
        flush_every=MAPILLARY_FLUSH_EVERY,
        progress_every=MAPILLARY_PROGRESS_EVERY,
        skip_missing_coords=MAPILLARY_SKIP_MISSING_COORDS,
        max_workers=MAPILLARY_MAX_WORKERS,
    )
    cached_segments = cache['OBJECTID'].nunique() if not cache.empty and 'OBJECTID' in cache.columns else 0
    print(f'Updated Mapillary evidence cache: {VRU_EVIDENCE_PATH}')
    print(f'Cached coordinate-eligible segment(s): {cached_segments}')
    display(cache.head())
elif VRU_EVIDENCE_PATH.exists():
    print(f'Using existing Mapillary evidence cache: {VRU_EVIDENCE_PATH}')
    display(pd.read_csv(VRU_EVIDENCE_PATH).head())
else:
    print('Mapillary cache not built. Set BUILD_MAPILLARY_CACHE = True to create it from the API.')

vru_segment_check = build_segment_vru_check(prioritized, VRU_EVIDENCE_PATH)
if vru_segment_check.empty:
    print('VRU segment check will run after prioritized segments are available.')
else:
    review_cols = [c for c in [
        'OBJECTID', 'english_ro', 'RoadClass', 'LandUse', 'priority_score',
        'query_lon', 'query_lat', 'mapillary_has_coverage', 'mapillary_image_count',
        'vru_presence_status', 'vru_signal_count', 'vru_check_action',
        'mapillary_images_cached', 'mapillary_api_value_count', 'mapillary_api_values', 'mapillary_api_errors',
    ] if c in vru_segment_check.columns]
    sort_col = 'priority_score' if 'priority_score' in vru_segment_check.columns else 'OBJECTID'
    display(vru_segment_check.sort_values(sort_col, ascending=False)[review_cols].head(50))
    display(vru_status_summary(vru_segment_check))



Mapillary coordinate-eligible segments: 11544 of 11544 target segment(s)
Mapillary token loaded
Mapillary run config: target_segments=11544, batch_size=None, images_per_segment=1, thumbnails=False, image_detections=False, workers=6
Mapillary evidence progress: no new coordinate-eligible segment(s) to process
Updated Mapillary evidence cache: /Users/markjaysondoma/Documents/safe-speed-adb-challenge/data/mapillary_vru_evidence.csv
Cached coordinate-eligible segment(s): 11544


,OBJECTID,english_ro,RoadClass,LandUse,priority_score,query_lon,query_lat,mapillary_has_coverage,mapillary_image_count,mapillary_images_cached,mapillary_image_rank,mapillary_image_id,mapillary_captured_at,mapillary_compass_angle,mapillary_is_pano,mapillary_image_lon,mapillary_image_lat,mapillary_thumbnail_url,sidewalk_present,crossing_present,school_context,market_context,transit_stop_context,pedestrian_activity_visible,mapillary_api_value_count,mapillary_feature_error,mapillary_api_values,school_or_market_context,speed_sign_visible,notes,mapillary_error,mapillary_api_errors
0,15807,Udon Ratthaya Expressway,motorway,URBAN,0.99012,100.543238,13.901830,1,481,3.0,1.0,9.611334e+14,1.770182e+12,195.504362,False,100.543399,13.901795,NaN,0.0,0.0,0.0,1.0,0.0,1.0,179.0,NaN,complementary--chevron-left--g1;information--g...,NaN,NaN,NaN,NaN,NaN
1,15807,Udon Ratthaya Expressway,motorway,URBAN,0.99012,100.543238,13.901830,1,481,3.0,2.0,1.492925e+15,1.776872e+12,17.093184,False,100.543077,13.901211,NaN,0.0,0.0,0.0,1.0,0.0,1.0,179.0,NaN,complementary--chevron-left--g1;information--g...,NaN,NaN,NaN,NaN,NaN
2,15807,Udon Ratthaya Expressway,motorway,URBAN,0.99012,100.543238,13.901830,1,481,3.0,3.0,1.903897e+15,1.770182e+12,191.843183,False,100.543292,13.901373,NaN,0.0,0.0,0.0,1.0,0.0,1.0,179.0,NaN,complementary--chevron-left--g1;information--g...,NaN,NaN,NaN,NaN,NaN
3,16367,Prachim Ratthaya Expressway,motorway,URBAN,0.98711,100.425830,13.799194,1,59,3.0,1.0,1.079423e+15,1.553013e+12,96.278688,False,100.425784,13.799256,NaN,0.0,0.0,0.0,0.0,0.0,0.0,77.0,NaN,construction--flat--driveway;object--banner;ob...,NaN,NaN,NaN,NaN,NaN
4,16367,Prachim Ratthaya Expressway,motorway,URBAN,0.98711,100.425830,13.799194,1,59,3.0,2.0,3.373410e+14,1.553013e+12,92.444456,False,100.425221,13.799303,NaN,0.0,0.0,0.0,0.0,0.0,0.0,77.0,NaN,construction--flat--driveway;object--banner;ob...,NaN,NaN,NaN,NaN,NaN


,OBJECTID,english_ro,RoadClass,LandUse,priority_score,query_lon,query_lat,mapillary_has_coverage,mapillary_image_count,vru_presence_status,vru_signal_count,vru_check_action,mapillary_images_cached,mapillary_api_value_count,mapillary_api_values,mapillary_api_errors
1231,15807,Udon Ratthaya Expressway,motorway,URBAN,0.990120,100.543238,13.901830,1,481,needs_mapillary_or_manual_review,0.0,Review Mapillary API values or label VRU evide...,3,537.0,complementary--chevron-left--g1;information--g...,NaN
1481,16367,Prachim Ratthaya Expressway,motorway,URBAN,0.987110,100.425830,13.799194,1,59,needs_mapillary_or_manual_review,0.0,Review Mapillary API values or label VRU evide...,3,231.0,construction--flat--driveway;object--banner;ob...,NaN
1482,16369,Prachim Ratthaya Expressway,motorway,URBAN,0.984893,100.549918,13.811253,1,3663,needs_mapillary_or_manual_review,0.0,Review Mapillary API values or label VRU evide...,3,1137.0,complementary--chevron-right--g1;construction-...,NaN
79,178,Buraphawithi Expressway,motorway,URBAN,0.984611,101.004886,13.488935,1,526,needs_mapillary_or_manual_review,0.0,Review Mapillary API values or label VRU evide...,3,591.0,complementary--chevron-right--g1;object--banne...,NaN
5945,36078,Phetkasem Road,trunk,RURAL,0.984031,98.426137,8.333841,1,21,needs_mapillary_or_manual_review,0.0,Review Mapillary API values or label VRU evide...,3,30.0,general--traffic-sign--g1;object--sign--advert...,NaN
2104,17519,Asian Highway,trunk,URBAN,0.983472,100.352010,14.991996,1,237,needs_mapillary_or_manual_review,0.0,Review Mapillary API values or label VRU evide...,1,123.0,complementary--maximum-speed-limit-30--g1;info...,NaN
5629,35515,Chalong Rat Expressway,motorway,URBAN,0.982662,100.689098,13.897436,1,1524,needs_mapillary_or_manual_review,0.0,Review Mapillary API values or label VRU evide...,1,243.0,complementary--chevron-left--g1;complementary-...,NaN
1232,15808,Udon Ratthaya Expressway,motorway,URBAN,0.982298,100.556997,14.145231,1,198,needs_mapillary_or_manual_review,0.0,Review Mapillary API values or label VRU evide...,1,271.0,complementary--texts-two-lines--g1;constructio...,NaN
4468,32459,,motorway,URBAN,0.982136,100.671476,13.655475,1,1882,needs_mapillary_or_manual_review,0.0,Review Mapillary API values or label VRU evide...,1,206.0,complementary--one-direction-left--g1;general-...,NaN
2085,17498,Asian Highway,trunk,URBAN,0.980219,100.313632,15.075329,1,241,needs_mapillary_or_manual_review,0.0,Review Mapillary API values or label VRU evide...,1,106.0,construction--flat--driveway;information--gene...,NaN


,status,segments
0,needs_mapillary_or_manual_review,7415
1,unknown_no_mapillary_coverage,4129


## 9. Risk Analysis


In [9]:
VRU_EXPOSURE_WEIGHTS = DEFAULT_VRU_EXPOSURE_WEIGHTS.copy()
# Edit these values to tune the VRU exposure model.
# school_context and market_context fall back to school_or_market_context when separate labels are not available.
VRU_EXPOSURE_WEIGHTS.update({
    'is_urban_land_use': 0.20,
    'road_class_vru_weight': 0.12,
    'traffic_component': 0.13,
    'sidewalk_present': 0.08,
    'crossing_present': 0.12,
    'school_context': 0.07,
    'market_context': 0.07,
    'transit_stop_context': 0.06,
    'pedestrian_activity_visible': 0.10,
    'has_street_image_coords': 0.03,
    'has_mapillary_evidence': 0.02,
})

prioritized = (
    add_vru_exposure_features(prioritized, VRU_EVIDENCE_PATH, vru_weights=VRU_EXPOSURE_WEIGHTS)
    if not prioritized.empty
    else prioritized
)

if prioritized.empty:
    print('Risk analysis will run after data is loaded.')
else:
    display(pd.DataFrame(VRU_EXPOSURE_WEIGHTS.items(), columns=['factor', 'weight']))
    vru_cols = [c for c in [
        'safety_priority_score', 'priority_score', 'vru_exposure_score', 'OBJECTID', 'english_ro',
        'RoadClass', 'LandUse', 'is_urban_land_use', 'road_class_vru_weight', 'traffic_component',
        'sidewalk_present_factor', 'crossing_present_factor', 'school_context_factor', 'market_context_factor',
        'transit_stop_context_factor', 'pedestrian_activity_visible_factor',
        'manual_vru_indicator_score', 'has_manual_vru_evidence', 'has_mapillary_evidence',
        'mapillary_image_count', 'has_street_image_coords'
    ] if c in prioritized.columns]
    display(prioritized.sort_values('safety_priority_score', ascending=False)[vru_cols].head(100))

    out_path = export_priority_segments(prioritized, OUTPUT_DIR)
    if out_path is not None:
        print(f'Wrote {out_path}')


,factor,weight
0,is_urban_land_use,0.20
1,road_class_vru_weight,0.12
2,traffic_component,0.13
3,sidewalk_present,0.08
4,crossing_present,0.12
5,school_context,0.07
6,market_context,0.07
7,transit_stop_context,0.06
8,pedestrian_activity_visible,0.10
9,has_street_image_coords,0.03


,safety_priority_score,priority_score,vru_exposure_score,OBJECTID,english_ro,RoadClass,LandUse,is_urban_land_use,road_class_vru_weight,traffic_component,sidewalk_present_factor,crossing_present_factor,school_context_factor,market_context_factor,transit_stop_context_factor,pedestrian_activity_visible_factor,manual_vru_indicator_score,has_manual_vru_evidence,has_mapillary_evidence,mapillary_image_count,has_street_image_coords
4901,0.858202,0.980204,0.492196,34226,Srinagarindra-Rom Klao Road,secondary,URBAN,1.0,1.00,0.939969,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,168,1.0
2104,0.850156,0.983472,0.450209,17519,Asian Highway,trunk,URBAN,1.0,0.60,0.986227,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,237,1.0
1222,0.847435,0.966836,0.489234,15760,Ratchaphruek Road,secondary,URBAN,1.0,1.00,0.917186,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,748,1.0
2085,0.847263,0.980219,0.448396,17498,Asian Highway,trunk,URBAN,1.0,0.60,0.972280,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,241,1.0
4653,0.845303,0.969952,0.471358,33833,Suranarai Road,primary,URBAN,1.0,0.85,0.918139,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,199,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6336,0.809671,0.934360,0.435604,36695,Highway 42,trunk,URBAN,1.0,0.60,0.873874,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,16,1.0
1836,0.809233,0.935391,0.430761,17115,Chaiyaphum-Si Khio Road,trunk,URBAN,1.0,0.60,0.836625,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,166,1.0
5101,0.808930,0.929738,0.446505,34549,Sukhumvit Road,trunk,URBAN,1.0,0.60,0.957727,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,706,1.0
2073,0.808831,0.930102,0.445018,17484,Asian Highway,trunk,URBAN,1.0,0.60,0.946292,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,11,1.0


Wrote /Users/markjaysondoma/Documents/safe-speed-adb-challenge/outputs/safe_speed_priority_segments.csv


## 10. Visualization

Blank for now.


## 11. Policy Generation

Blank for now.
